# Tourisim Uncertainty Index Code

## System Requirements

In [3]:
import sys
import platform

print("===== ENVIRONMENT SUMMARY =====")

# Python & OS
print(f"Python: {sys.version.split()[0]}")
print(f"OS: {platform.system()} {platform.release()}")

# Core libraries
import spacy
import pandas
import fitz  # PyMuPDF

print(f"spaCy: {spacy.__version__}")
print(f"pandas: {pandas.__version__}")
print(f"PyMuPDF: {fitz.__doc__.split()[1]}")

# spaCy model
try:
    nlp = spacy.load("en_core_web_lg")
    print(f"spaCy model (en_core_web_lg): {nlp.meta.get('version')}")
except:
    print("spaCy model: NOT INSTALLED")

# Optional: CPU & RAM (short)
try:
    import psutil
    print(f"CPU cores: {psutil.cpu_count(logical=True)}")
    print(f"RAM (GB): {round(psutil.virtual_memory().total / (1024**3))}")
except:
    pass

===== ENVIRONMENT SUMMARY =====
Python: 3.12.3
OS: Windows 11
spaCy: 3.8.11
pandas: 2.2.2
PyMuPDF: 1.25.4:
spaCy model (en_core_web_lg): 3.7.1
CPU cores: 12
RAM (GB): 32


In [3]:
# pip install spacy

# Import Libraries

In [4]:
import spacy
import re
import os
import fitz  # PyMuPDF
import pandas as pd
import warnings

# Suppress unnecessary warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

# Main Code

In [1]:
# Load spaCy language model
nlp = spacy.load("en_core_web_lg")

# Define uncertainty terms
uncertainty_related_words = [
    "alterations", "ambiguity", "anomalies", "anomaly", "attainability", "complexity", "complexities", "confounding",
    "confusion", "contestation", "contradictions", "crisis", "delays", "default", "deviations", "disorientation",
    "disparity", "disputable", "disorder", "dissonance", "doubt", "divergence", "divergences", "elections",
    "embargoes", "erosion", "exposures", "fluctuations", "flux", "fragility", "fragmentation", "heterogeneity",
    "hazard", "hazardous", "imbalance", "indecision", "indeterminacy", "inevitability", "instabilities",
    "instability", "instable", "instinctiveness", "irregularity", "limitation", "misalignment", "misjudgment",
    "mistrust", "nonlinearity", "oscillations", "paradox", "perplexity", "precariousness", "problematic",
    "probabilities", "puzzling", "questionable", "recessions", "risk", "riskiness", "sanctions", "scenarios",
    "security", "shifting", "shocks", "speculation", "tensions", "thresholds", "turbulence", "turmoil",
    "unachievable", "unattainable", "unavailability", "uncertain", "uncertainty", "uncertainties", "unclear",
    "unfixed", "unforeseen", "unpredictable", "unreliable", "unresolved", "unstable", "unknown",
    "variability", "volatile", "volatility", "vulnerability"
]

# Define tourism-related phrases
tourism_related_phrases = [
    "tourist revenue", "tourism spending", "accommodation costs", "exchange rates",
    "subsidized travel", "subsidised travel", "budget travel", "luxury tourism", "tourism demand",
    "airline pricing", "hospitality sector", "foreign earnings", "revenue decline", "travel expenses",
    "low-cost travel", "charter flights", "cruise tourism", "hotel occupancy", "budget airlines",
    "tourism industry", "hotel chains", "group discounts", "booking platforms", "visitor arrivals",
    "airport fees", "consumer behavior", "consumer behaviour", "travel refunds", "business tourism",
    "tourism taxation", "sales tax", "destination costs", "tourist debt", "hotel rooms", "overnight stays",
    "occupancy rate", "travel insurance", "tourism startups", "booking cancellations", "tourism dependency",
    "package deals", "market competition", "industry disruption", "discount programs", "tourism branding",
    "tourist savings", "employment generation", "transportation costs", "travel agents", "foreign investments",
    "destination pricing", "exchange volatility", "market segmentation", "direct tourism", "visitor expenditure",
    "corporate travel", "travel packages", "seasonal pricing", "inflation rates", "demand elasticity",
    "visitor income", "tourism credits", "visitor statistics", "tourism analysis", "travel logistics",
    "price fluctuations", "advertising costs", "infrastructure costs", "resource usage", "domestic tourism",
    "international arrivals", "travel economics", "operational costs", "visitor trends", "cultural heritage",
    "local traditions", "visitor behavior", "visitor behaviour", "social media", "host relations",
    "tourism protests", "festival attendance", "public health", "tourism equality", "accessible tourism",
    "heritage preservation", "volunteer tourism", "local communities", "social impact", "tourist satisfaction",
    "event tourism", "lifestyle changes", "visitor expectations", "tourism dynamics", "community benefits",
    "social cohesion", "tourism employment", "visitor diversity", "local hospitality", "cultural exchanges",
    "urban tourism", "social tensions", "social awareness", "tourism traditions", "domestic travel",
    "social capital", "family tourism", "regional identity", "local support", "tourism education",
    "community resilience", "cultural integration", "multicultural events", "population mobility",
    "public gatherings", "local lifestyles", "social initiatives", "cross-cultural dialogue",
    "heritage tourism", "social interactions", "community networks", "visitor happiness", "social inclusion",
    "cultural tourism", "festival culture", "travel satisfaction", "carbon emissions", "marine pollution",
    "green tourism", "wildlife protection", "habitat loss", "resource scarcity", "natural disasters",
    "water pollution", "air quality", "climate risks", "ecosystem damage", "over-tourism effects",
    "biodiversity loss", "renewable energy", "conservation efforts", "environmental degradation",
    "waste management", "protected areas", "sustainable travel", "tourism ecology", "forest depletion",
    "soil erosion", "climate adaptation", "coastal tourism", "pollution control", "land preservation",
    "wildlife tourism", "renewable solutions", "energy efficiency", "carbon reduction", "tourism footprints",
    "beach erosion", "glacier melting", "clean energy", "natural reserves", "emission controls",
    "sea level rise", "sustainable beaches", "nature reserves", "renewable practices", "ecotourism development",
    "air pollution", "green infrastructure", "eco-certifications", "recycling programs", "wildlife preservation",
    "climate mitigation", "marine conservation", "renewable resources", "drought management", "land usage",
    "glacier tourism", "solar energy", "clean technology", "pollution reduction", "habitat restoration",
    "environmental risks", "clean water", "environmental awareness", "natural parks", "urban green spaces",
    "sustainable consumption", "zero emissions", "responsible travel", "carbon neutrality",
    "eco-friendly tourism", "environmental audits", "habitat tourism", "environmental policies",
    "wildlife observation", "rainforest tourism", "environmental equity", "marine sustainability",
    "green operations", "sustainable reserves", "carbon-neutral travel", "travel bans", "visa policies",
    "political stability", "border closures", "terrorism threats", "security protocols", "trade sanctions",
    "visa waiver program", "regional conflicts", "government policies", "immigration laws",
    "tourism sanctions", "geopolitical risks", "diplomatic relations", "regulatory frameworks",
    "tourism embargos", "crisis management", "policy uncertainty", "conflict zones", "policy reforms",
    "international treaties", "safety advisories", "peace tourism", "foreign diplomacy", "national security",
    "political unrest", "cultural diplomacy", "border security", "visa restrictions", "emergency measures",
    "bilateral agreements", "geopolitical tensions", "conflict resolution", "election impact",
    "government interventions", "tourism subsidies", "regional cooperation", "state regulations",
    "cross-border travel", "humanitarian aid", "military conflicts", "political disputes",
    "governance reforms", "foreign aid policies", "national strategies", "diplomatic disputes",
    "peace agreements", "sanction policies", "political influences", "geopolitical alliances",
    "trade agreements", "legislative changes", "migration policies", "refugee crises",
    "political embargoes", "diplomatic initiatives", "global cooperation", "political accountability",
    "leadership strategies", "trade barriers", "strategic diplomacy", "legal frameworks",
    "security escalations", "regional stability"
]

# Context window size in terms of number of words
context_window_size = 10

# Folder path containing PDF reports
folder_path = r"C\....\Sample Reports" # <-- Change with your folder's path
 
# Create a list to store results
results_list = []

def generate_ngrams(words, n):
    return [' '.join(words[i:i + n]) for i in range(len(words) - n + 1)]

# Iterate through files in the folder
for filename in os.listdir(folder_path):
    if filename.endswith(".pdf"):
        file_path = os.path.join(folder_path, filename)

        with fitz.open(file_path) as pdf_file:
            text = ""
            for page in pdf_file:
                text += page.get_text()

            # Preprocess text for cleaner analysis
            text = re.sub(r'\s+', ' ', text)
            text = re.sub(r'\n', ' ', text)
            text = ''.join(ch for ch in text if ch.isprintable())
            text = text.lower()

            # Tokenize text using spaCy
            doc = nlp(text)
            tokens = [token.text for token in doc if not token.is_stop and token.is_alpha]

            # Get total word count after cleaning
            total_word_count = len(tokens)

            # Initialize counter
            tourism_related_keywords_number = 0

            # Check for uncertainty-related words and context
            for i, token in enumerate(tokens):
                if token in uncertainty_related_words:
                    # Define context window
                    start_index = max(0, i - context_window_size)
                    end_index = min(len(tokens), i + context_window_size + 1)

                    # Generate bi-grams and tri-grams only for tourism-related phrases
                    bigrams = generate_ngrams(tokens[start_index:end_index], 2)
                    trigrams = generate_ngrams(tokens[start_index:end_index], 3)

                    # Check if tourism-related phrases exist in the context window
                    if any(phrase in bigrams or phrase in trigrams for phrase in tourism_related_phrases):
                        tourism_related_keywords_number += 1

            # Add results to the list
            results_list.append({
                "File": filename,
                "Total Words Number": total_word_count,
                "Tourism Related Uncertainty Keywords Number": tourism_related_keywords_number
            })

# Convert results list to DataFrame
results = pd.DataFrame(results_list)

# Get current working directory
current_dir = os.getcwd()

# Construct output file path
excel_file_path = os.path.join(current_dir, "Tourism_Uncertainty_Keywords_Number.xlsx") # <-- Output Excel file's name

# Save results to Excel file
results.to_excel(excel_file_path, index=False)

print("Excel file successfully created:", excel_file_path)
print(results)

Excel file successfully created: C:\Users\ismet\OneDrive - University of Strathclyde\Desktop\TUI - Duzeltme - 18 March 2026\18 March 2026 - Revision\GitHub\Tourism_Uncertainty_Keywords_Number.xlsx
               File  Total Words Number  Tourism Uncertainty Keywords Number
0  2005 - A - 1.pdf                1898                                    1
1  2005 - B - 2.pdf                1883                                    1
2  2005 - C - 3.pdf               10027                                    1


In [ ]:
# This code writes these results in an Excel file in your working folder 